# Universal Domain Gathering Pipeline

This notebook merges the optimized performance of `chatbot_pipeline.py` with the robust fallbacks of your LangGraph notebook.

In [18]:
import re
import json
import string
from collections import Counter
import pandas as pd
import os
from pathlib import Path

# Optional: NLTK for lemmatization
try:
    from nltk.stem import WordNetLemmatizer
    import nltk
    nltk.download('wordnet', quiet=True)
    _lemmatizer = WordNetLemmatizer()
    USE_NLTK = True
except ImportError:
    _lemmatizer = None
    USE_NLTK = False

def clean_token(token: str) -> str:
    """Normalize a single token: lowercase, strip, and lemmatize if NLTK is available."""
    token = str(token).strip().lower()
    if USE_NLTK and _lemmatizer and len(token) > 2:
        return _lemmatizer.lemmatize(token)
    return token

In [19]:
class UltimateDomainVocabularyBuilder:
    """The Best of Both Worlds: Robust, Optimized, Smart Domain Extraction."""

    def __init__(self, fashion_df):
        self.fashion_df = fashion_df
        self.vocabulary = {
            'brands': set(), 'colors': set(), 'products': set(),
            'materials': set(), 'sizes': set(), 'styles': set(),
            'gender': set(), 'all_tokens': set()
        }

    def _clean_token(self, token):
        return clean_token(token)

    def extract_from_structured_columns(self):
        """Extracts using resilient substring matching for columns."""
        print("\n[1] Extracting from structured columns...")
        
        column_mapping = {
            'color': 'colors', 'colour': 'colors', 'size': 'sizes', 
            'material': 'materials', 'style': 'styles', 'usage': 'styles', 
            'gender': 'gender', 'category': 'products', 'type': 'products'
        }

        for col in self.fashion_df.columns:
            col_lower = str(col).lower()
            for key, vocab_key in column_mapping.items():
                if key in col_lower:
                    tokens = {
                        self._clean_token(t) for val in self.fashion_df[col].dropna()
                        for t in str(val).split(',') if len(str(t).strip()) > 1
                    }
                    self.vocabulary[vocab_key].update(tokens)
                    print(f"  ✓ Extracted {len(tokens)} {vocab_key} from column '{col}'")
                    break

    def extract_brands(self):
        """Specific dynamic extraction for Brand columns."""
        print("\n[2] Mining for specific brand columns...")
        brand_cols = [c for c in self.fashion_df.columns if 'brand' in str(c).lower()]
        for col in brand_cols:
            brands = {str(b).strip().lower() for b in self.fashion_df[col].dropna() if len(str(b).strip()) > 1}
            self.vocabulary['brands'].update(brands)
            print(f"  ✓ Extracted {len(brands)} brands from '{col}'")

    def extract_products_from_titles(self):
        """Advanced logic to find missing products focusing on head nouns and excluding knowns."""
        print("\n[3] Mining titles for additional product nouns...")

        cats_to_exclude = ['brands', 'colors', 'gender', 'styles', 'materials']
        exclusions = set().union(*(self.vocabulary[cat] for cat in cats_to_exclude))
        noise = {'navy', 'golden', 'length', 'lifestyle', 'coloured', 'kids', 'men', 'women', 'canvas', 'leather', 'printed'}

        word_pattern = re.compile(r'\b\w+\b')
        potential_products = []

        for title in self.fashion_df['ProductTitle'].dropna():
            words = word_pattern.findall(str(title).lower())
            if not words: continue
            
            candidates = [words[-1]]
            if len(words) > 1: 
                candidates.append(words[-2])

            for word in candidates:
                word_clean = self._clean_token(word)
                # Ensure no pure numbers sneak in before the LLM
                if len(word_clean) > 2 and word_clean not in exclusions and word_clean not in noise and not word_clean.isnumeric():
                    potential_products.append(word_clean)

        word_counts = Counter(potential_products)
        common_nouns = {word for word, count in word_counts.items() if count >= 3}

        self.vocabulary['products'].update(common_nouns)
        print(f"  ✓ Added {len(common_nouns)} highly-probable product types from titles.")

    def inject_hardcoded_failsafe(self):
        """Ensure crucial baseline descriptors always exist, protecting against partial data or LLM failure."""
        print("\n[4] Injecting baseline failsafe descriptors...")
        gender_terms = {'men', 'mens', "men's", 'women', 'womens', "women's", 'unisex',
                       'boys', 'boy', "boy's", 'girls', 'girl', "girl's", 'kids', "kids'",
                       'infant', 'baby', 'toddler', 'children', 'kidswear', 'son', 'daughter'}
        self.vocabulary['gender'].update(self._clean_token(t) for t in gender_terms)
        print(f"  ✓ Injected {len(gender_terms)} core routing terms for gender.")

    def add_llm_descriptors(self, genai_client=None):
        """Dynamic domain review and expansion via modern Google GenAI SDK, strictly enforcing JSON format."""
        if not genai_client:
            print("\n[5] Skipping LLM augmentation: no LLM client provided.")
            return

        print("\n[5] Calling Gemini LLM for domain sanitization and expansion (Strict JSON mode)...")
        
        # EXCLUDED BRANDS entirely to protect the raw CSV data
        current_state = {
            'gender': list(self.vocabulary['gender']),
            'colors': list(self.vocabulary['colors']),
            'styles': list(self.vocabulary['styles']),
            'materials': list(self.vocabulary['materials']),
            'products': list(self.vocabulary['products'])
        }
        state_json = json.dumps(current_state, indent=2)
        
        prompt = f"""You are an elite Fashion Domain Data Scientist.
I am providing you with a raw, extracted fashion dataset vocabulary in JSON format. It contains noisy and irrelevant terms, and is likely missing key generic fashion terms.

Your tasks:
1. REVIEW AND CLEAN: Prune each category. Remove words that are pure numbers, nonsensical, misspellings, or definitively NOT related to fashion clothing. For colors, products, materials, and styles, be strictly rigorous and permanently remove noise like 'printed', 'lifestyle', 'length', 'toprated', etc.
2. EXPAND: Add any fundamental fashion descriptors that are completely missing from each category to make it a globally comprehensive dictionary.

Return ONLY a valid JSON object strictly matching these 5 exact keys: 'gender', 'styles', 'materials', 'colors', 'products', each containing arrays of cleaned, lowercase python strings.

Here is the current raw vocabulary state limit output to json:
{state_json}"""

        try:
            from google.genai import types
            
            response = genai_client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.3,
                    max_output_tokens=8192
                )
            )
            llm_output = response.text
            llm_vocab = json.loads(llm_output)

            # EXCLUDED BRANDS from the target modification keys
            target_keys = ['gender', 'colors', 'styles', 'materials', 'products']
            print("  ✓ Received LLM Review...")
            
            for category in target_keys:  
                if category in llm_vocab and isinstance(llm_vocab[category], list):
                    original_len = len(self.vocabulary[category])
                    clean_terms = {self._clean_token(t) for t in llm_vocab[category] if len(str(t).strip()) > 1}
                    
                    self.vocabulary[category] = clean_terms

                    new_len = len(clean_terms)
                    diff = new_len - original_len
                    sign = "+" if diff >= 0 else ""
                    print(f"    - {category.capitalize()}: {original_len} -> {new_len} terms ({sign}{diff})")

        except Exception as e:
            print(f"  ⚠️ LLM domain review/expansion failed: {e}. Relying solely on CSV data.")

    def build_and_save(self, genai_client=None, filename="domain_vocabulary.json"):
        """Execute the full pipeline, flatten, and export the vocabulary."""
        print("\n" + "*"*50)
        print("BUILDING MIXED DOMAIN VOCABULARY")
        print("*"*50)

        self.extract_from_structured_columns()
        self.extract_brands()
        self.extract_products_from_titles()
        self.inject_hardcoded_failsafe()
        self.add_llm_descriptors(genai_client)

        # Consolidate 'all_tokens' for your router into purely lowercase generic terms
        for cat, terms in self.vocabulary.items():
            if cat != 'all_tokens':
                self.vocabulary['all_tokens'].update({str(t).strip().lower() for t in terms})

        # Convert sets to sorted lists for clean JSON rendering
        final_vocab = {k: sorted(list(v)) for k, v in self.vocabulary.items()}
        
        # Detailed print summary from the Notebook logic
        print("\n📊 Final Domain Vocabulary Summary")
        for category, tokens in final_vocab.items():
            if category != 'all_tokens' and tokens:
                print(f"  {category:15s}: {len(tokens):4d} terms")
        print(f"\n✓ TOTAL UNIQUE TOKENS (all_tokens): {len(final_vocab['all_tokens'])}")

        # Export for LangGraph Router safely
        try:
            dir_name = os.path.dirname(filename)
            if dir_name: 
                os.makedirs(dir_name, exist_ok=True)
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(final_vocab, f, indent=4, ensure_ascii=False)
            print(f"\n✅ Domain explicitly saved to '{filename}' for router.")
        except Exception as e:
            print(f"\n⚠️ Failed to export '{filename}': {e}")

        return final_vocab


In [20]:
# Example usage block explicitly using your Gemini API key via .env
!pip install google-genai python-dotenv pandas
import os
from pathlib import Path
from google import genai
from dotenv import load_dotenv

# Look for a local .env file where the notebook is running
load_dotenv(override=True)

# Unset VSCode Colab proxy interceptors
for key in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY']:
    os.environ.pop(key, None)

try:
    # Note: Make sure the path to your CSV is correct
    # Allow environment overriding, defaulting to Colab's relative path
    _CSV_PATH = Path(os.environ.get("CSV_PATH", "fashion.csv"))
    _JSON_PATH = Path(os.environ.get("JSON_PATH", "files/help.json"))

    print(f"Reading dataset from: {_CSV_PATH}")
    
    # This will throw FileNotFoundError if the file doesn't exist at the exact path
    df = pd.read_csv(_CSV_PATH) 
    
    # If read_csv succeeds, we correctly define builder
    builder = UltimateDomainVocabularyBuilder(df)
    
    # Fetch your specific GOOGLE_API_KEY from environment variables (.env)
    gemini_key = os.environ.get("GOOGLE_API_KEY")
    
    if not gemini_key:
        print("⚠️ GOOGLE_API_KEY not found in environment or .env file.")
        print("Are you sure you added it exactly as GOOGLE_API_KEY=... in your .env file?")
        # Running without LLM if no key is provided
        vocab = builder.build_and_save(filename=str(_JSON_PATH))
    else:
        print("Initializing Google Gemini client...")
        # Native V2 SDK initialization
        client = genai.Client(api_key=gemini_key)
        
        vocab = builder.build_and_save(genai_client=client, filename=str(_JSON_PATH))
    
except FileNotFoundError:
    print("="*50)
    print(f"❌ ERROR: The file '{_CSV_PATH}' was not found!")
    print("="*50)
    print("Please check your Colab file explorer (folder icon on the left).")
    print("If you uploaded 'fashion.csv' directly to the main Colab directory, change the path above to:")
    print("_CSV_PATH = Path('fashion.csv')")
    print("If you uploaded the whole 'chatbot' folder, make sure the lowercase/uppercase structure perfectly matches.")
except Exception as e:
    print(f"An error occurred: {e}")

Reading dataset from: fashion.csv
Initializing Google Gemini client...

**************************************************
BUILDING MIXED DOMAIN VOCABULARY
**************************************************

[1] Extracting from structured columns...
  ✓ Extracted 4 gender from column 'Gender'
  ✓ Extracted 2 products from column 'Category'
  ✓ Extracted 9 products from column 'SubCategory'
  ✓ Extracted 31 products from column 'ProductType'
  ✓ Extracted 39 colors from column 'Colour'
  ✓ Extracted 6 styles from column 'Usage'
  ✓ Extracted 7 sizes from column 'Size'

[2] Mining for specific brand columns...
  ✓ Extracted 84 brands from 'BrandName'

[3] Mining titles for additional product nouns...
  ✓ Added 57 highly-probable product types from titles.

[4] Injecting baseline failsafe descriptors...
  ✓ Injected 22 core routing terms for gender.

[5] Calling Gemini LLM for domain sanitization and expansion (Strict JSON mode)...
  ✓ Received LLM Review...
    - Gender: 19 -> 17 terms (